# 🎤 הפרדת שירה באיכות אולפן — לאפליקציית הקריוקי

המחברת הזו מריצה **בחינם, על GPU של Google**, את מודלי הפרדת השירה החדישים ביותר שקיימים בקוד פתוח (משפחת **RoFormer** — הדור שאחרי Demucs/MDX, הרמה הקרובה ביותר לשירותים בתשלום כמו lalal.ai).

### לפני שמתחילים — פעם אחת:
1. בתפריט למעלה: **Runtime ← Change runtime type ← T4 GPU ← Save**
2. מריצים את התאים לפי הסדר (לחיצה על ▶ ליד כל תא)

### הזרימה:
התקנה (2-3 דק', פעם אחת לכל חיבור) ← העלאת שיר ← הפרדה (2-4 דק' לשיר) ← הורדת הפלייבק ← טעינה כשיר באפליקציית הקריוקי 🎶

In [ ]:
#@title 1️⃣ התקנה (2-3 דקות, פעם אחת) { display-mode: "form" }
!pip install -q "audio-separator[gpu]"
import torch
if torch.cuda.is_available():
    print("✅ GPU פעיל:", torch.cuda.get_device_name(0))
else:
    print("⚠️ אין GPU! למעלה: Runtime → Change runtime type → T4 GPU, ואז להריץ את התא הזה שוב")

In [ ]:
#@title 2️⃣ העלאת שיר (אפשר כמה קבצים) { display-mode: "form" }
from google.colab import files
uploaded = files.upload()
song_files = list(uploaded.keys())
print("\n🎵 הועלו:", song_files)

In [ ]:
#@title 3️⃣ הפרדה { display-mode: "form" }
#@markdown בחרו מודל (ברירת המחדל מצוינת לפלייבקים — "Bleedless" = מינימום דליפת שירה):
model = "mel_band_roformer_kim_ft2_bleedless_unwa.ckpt" #@param ["mel_band_roformer_kim_ft2_bleedless_unwa.ckpt", "mel_band_roformer_kim_ft2_unwa.ckpt", "vocals_mel_band_roformer.ckpt"]

from audio_separator.separator import Separator
import os

separator = Separator(output_format="MP3", output_bitrate="320k", output_dir="/content/output")
print("⬇️ מוריד את המודל (פעם אחת לכל חיבור)…")
separator.load_model(model_filename=model)

all_outputs = []
for song in song_files:
    print(f"\n🎛️ מפריד: {song} …")
    outs = separator.separate(song)
    outs = [os.path.join("/content/output", o) if not os.path.isabs(o) else o for o in outs]
    all_outputs.extend(outs)
    for o in outs:
        print("   ✔", os.path.basename(o))
print("\n✅ סיימנו! עוברים לתא הבא להורדה.")

In [ ]:
#@title 4️⃣ הורדת התוצאות { display-mode: "form" }
#@markdown הקובץ עם **(Instrumental)** בשם = הפלייבק. הקובץ עם (Vocals) = השירה בלבד.
from google.colab import files
for o in all_outputs:
    files.download(o)

## 🎤 ומה עכשיו? חוזרים לאפליקציית הקריוקי

1. פותחים את [אפליקציית הקריוקי](https://yanivmoyal9.github.io/workout-tracker/karaoke.html)
2. **➕ הוספת שיר** ← בוחרים את קובץ ה-**(Instrumental)** שהורדתם
3. ממלאים שם ואמן ← 🔎 חיפוש מילים מסונכרנות ← שמירה
4. שרים! 🎶 (הפלייבק כבר נקי — את "הסרת שירה" באפליקציה משאירים כבויה, וטון/מהירות/מיקרופון עובדים כרגיל)

### טיפים
- השוו מודלים: אם נשארו שאריות, נסו את המודל השני ברשימה בתא 3 והריצו שוב
- Colab חינמי מנתק אחרי חוסר פעילות — להפרדה של עוד שירים מחר, פשוט מריצים הכל שוב
- אם שם מודל נכשל בהורדה (הרשימה מתעדכנת מדי פעם), הריצו בתא חדש: `!audio-separator --list_models` ובחרו שם עדכני